In [1]:
import duckdb
import gspread
import pandas as pd
# import numpy as np
from google.oauth2.service_account import Credentials

SCOPES = [
    # 'https://www.googleapis.com/auth/bigquery',
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets'
    # ,'https://www.googleapis.com/auth/devstorage.full_control'
]

credentials = Credentials.from_service_account_file(r'.secrets\ads-txt-validator-BQ.json', scopes=SCOPES)

In [2]:
sheet_client:gspread.Client = gspread.Client(auth=credentials)

def adhoc_gsheet_pull(worksheet_name:str):

        google_sheet_id = '1DLevo_k0LTfIHH7wL9n08NfYnpq_CbSF5RLNIMaIJU8'
        # worksheet_name = 'main_file'

        spreadsheet = sheet_client.open_by_key(google_sheet_id)

        worksheet = spreadsheet.worksheet(worksheet_name)

        list_of_dicts = worksheet.get_all_records()

        df = pd.DataFrame(list_of_dicts)

        try:
            if "ORG_ID" in df.columns:
                df = df.astype({"ORG_ID":"int64"}).astype({"ORG_ID":"string"})
        except Exception as e:
            print(f"{e=}")

        # print(df.dtypes)

        return df

main_file_df = adhoc_gsheet_pull(worksheet_name = 'main_file')
expected_ads_txt_df = adhoc_gsheet_pull(worksheet_name = 'ads_txt_lines')
# main_file_df

e=ValueError("invalid literal for int() with base 10: '494n1p1p': Error while type casting for column 'ORG_ID'")


In [3]:
raw_tbl = duckdb.read_parquet(r'data_output\2025-03-26\ads_txt_success_2025-03-26-15-22-01.parquet')
err_tbl = duckdb.read_parquet(r'data_output\2025-03-26\ads_txt_failure_2025-03-26-15-22-01.parquet')

In [ ]:
output_tbl = duckdb.sql(
    """
CREATE OR REPLACE MACRO SAFE_OFFSET(arr, idx) AS (
    CASE
        WHEN idx > 0 AND idx <= array_length(arr) THEN arr[idx]
        ELSE NULL
    END
);
WITH main_client_tbl AS (
  SELECT 
  Publisher AS publisher
  , ORG_ID AS org_id
  , CASE WHEN Inventory_Type = 'App - (app-ads.txt)' THEN 'app-ads.txt'
         WHEN Inventory_Type = 'Web - (ads.txt)' THEN 'ads.txt'
         ELSE Inventory_Type END AS inventory_type
  , Relationship_Type AS relationship_type
  , Account_Manager as account_manager_name
  , Account_Manager_Email as account_manager_email
  , Domain AS domain
  , Ad_Request AS ad_request
  , Revenue AS revenue
  FROM main_file_df
  WHERE 
  domain IS NOT NULL
  AND inventory_type IS NOT NULL
  AND org_id IS NOT NULL
  AND relationship_type IS NOT NULL
),

required_domains AS (
  SELECT DISTINCT domain, inventory_type, CONCAT(domain,'--',inventory_type) AS relevent_domains_key FROM main_client_tbl
),

required_bronze_data AS (
  SELECT LOWER(REGEXP_REPLACE(REPLACE(ads_txt_line, ' ', ''), '#.*','')) AS ads_txt_lines
  , domain
  , inventory_type
  , ads_txt_url AS ads_txt_url
  FROM raw_tbl
  WHERE 
  CONCAT(domain,'--',inventory_type) IN (SELECT DISTINCT relevent_domains_key from required_domains) 
  AND ads_txt_line is not null and  not starts_with(ads_txt_line, '#')
),

duplicated_rows_bronze_data AS (
  SELECT 
  ads_txt_lines
  , domain
  , inventory_type
  , ads_txt_url
  -- , execution_date 
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),1))  as ssp_domain
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),2))  as pub_id
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),3))  as relationship_type
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),4))  as ssp_id
  ,ROW_NUMBER() OVER (
      PARTITION BY 
        domain, 
        inventory_type, 
        ads_txt_url, 
        -- execution_date, 
        TRIM(SAFE_OFFSET(SPLIT(ads_txt_lines, ','),1)),  -- ssp_domain
        TRIM(SAFE_OFFSET(SPLIT(ads_txt_lines, ','),2)),  -- pub_id
        TRIM(SAFE_OFFSET(SPLIT(ads_txt_lines, ','),3))   -- relationship_type
      -- ORDER BY execution_date DESC  -- Keeping the latest record
    ) AS row_num
  FROM required_bronze_data
), 

cleaned_bronze_data AS (
  SELECT * FROM duplicated_rows_bronze_data WHERE row_num = 1
),

web_ads_txt_lines AS (
  SELECT LOWER(REGEXP_REPLACE(REPLACE(ads_txt_line, ' ', ''), '#.*','')) AS ads_txt_lines
  FROM expected_ads_txt_df
),

web_pubs AS (
  SELECT * FROM main_client_tbl WHERE inventory_type = 'ads.txt'
),

combined_ads_txt_lines_checking AS (
  SELECT publisher
  , org_id
  , inventory_type
  , relationship_type
  , account_manager_name
  , account_manager_email
  , domain
  , ad_request
  , revenue
  , REPLACE(ads_txt_lines, '[org_id],[relation_type]', LOWER(CONCAT(org_id,',',relationship_type))) AS ads_txt_lines
  FROM web_pubs CROSS JOIN web_ads_txt_lines 
),

cleaned_pubs_data AS (
  SELECT publisher
  , org_id
  , inventory_type
  , relationship_type AS relationship_type_given
  , account_manager_name
  , account_manager_email
  , domain
  , ad_request
  , revenue
  , ads_txt_lines 
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),1))  as ssp_domain
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),2))  as pub_id
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),3))  as relationship_type
  ,trim(SAFE_OFFSET(split(ads_txt_lines, ','),4))  as ssp_id
FROM combined_ads_txt_lines_checking),

error_tbl AS (SELECT
    domain,
    inventory_type,
    repr_error_msg
FROM (
    SELECT
        domain,
        inventory_type,
        repr_error_msg,
        ROW_NUMBER() OVER (
            PARTITION BY domain, inventory_type
            ORDER BY repr_error_msg
        ) AS rn
    FROM err_tbl
)
WHERE rn = 1
)



SELECT 
pubs_data.publisher
,pubs_data.org_id
,pubs_data.inventory_type
,pubs_data.relationship_type_given
,pubs_data.account_manager_name
,pubs_data.account_manager_email
,pubs_data.domain
,pubs_data.ad_request
,pubs_data.revenue
,pubs_data.ads_txt_lines
,pubs_data.ssp_domain
,pubs_data.pub_id
,pubs_data.relationship_type
,pubs_data.ssp_id
,scraped.ads_txt_lines AS ads_txt_lines_scraped
,scraped.domain AS domain_scraped
,scraped.inventory_type AS inventory_type_scraped
,scraped.ads_txt_url AS ads_txt_url_scraped
,scraped.ssp_domain AS ssp_domain_scraped
,scraped.pub_id AS pub_id_scraped
,scraped.relationship_type AS relationship_type_scraped
,scraped.ssp_id AS ssp_id_scraped
,CASE WHEN scraped.ads_txt_lines IS NULL THEN 'missing' ELSE 'present' END AS ads_txt_status
,CASE WHEN error_tbl.repr_error_msg IS NOT NULL THEN 'failure' WHEN scraped.ads_txt_lines IS NULL THEN 'missing' ELSE 'present' END AS ads_txt_status_v2
,error_tbl.repr_error_msg
FROM cleaned_pubs_data AS pubs_data LEFT JOIN cleaned_bronze_data AS scraped
ON pubs_data.pub_id = scraped.pub_id
AND pubs_data.inventory_type = scraped.inventory_type
AND pubs_data.domain = scraped.domain
AND pubs_data.ssp_domain = scraped.ssp_domain

LEFT JOIN error_tbl AS error_tbl
ON pubs_data.inventory_type = error_tbl.inventory_type
AND pubs_data.domain = error_tbl.domain

    """
)

In [5]:
output_tbl.show()

┌───────────────────────────┬──────────────────────────┬────────────────┬─────────────────────────┬─────────────────────────┬─────────────────────────┬───────────────────────────────┬────────────┬─────────┬───────────────────────────────────────────────────────────┬────────────┬──────────────────────────┬───────────────────┬──────────────────┬──────────────────────────────────────────────────────────────┬──────────────────────┬────────────────────────┬──────────────────────────────────────────┬────────────────────────┬──────────────────────────┬───────────────────────────┬──────────────────┬────────────────┬───────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│         publisher         │          org_id          │ inventory_type │ relationship_type_given │  account_manager_name   │  account_manager_email  │            domain             │ ad_request │ revenue │               

In [6]:
output_tbl.to_csv("didna_output_v3.csv",overwrite=True, header=True)
